<a href="https://colab.research.google.com/github/TCC540-a2026s2n11grupo1/TCC540-a2026s2n11grupo1-UNIVESP-2026/blob/main/Detec%C3%A7%C3%A3o_de_Fraudes_Financeiras_em_Grafos_com_GNNs_(GCNs).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Detecção de Fraudes Financeiras em Grafos com GNNs (GCNs)

Para a detecção de fraudes financeiras que envolvem relações complexas entre entidades (como contas, transações, usuários), a modelagem de dados como grafos é extremamente eficaz. Embora você tenha mencionado 'algoritmo CNN', Redes Convolucionais (CNNs) são mais comumente usadas para dados de grade (como imagens). Para dados de grafos, a abordagem equivalente e mais adequada são as **Redes Neurais Gráficas (GNNs)**, especificamente as **Redes Convolucionais Gráficas (GCNs)**.

Uma GCN é capaz de aprender representações (embeddings) para cada nó do grafo, levando em consideração suas próprias características e as características de seus vizinhos. Isso é crucial para detectar padrões de fraude que muitas vezes se manifestam através de conexões suspeitas.

Neste exemplo, vamos:
1.  **Instalar as bibliotecas necessárias**: `PyTorch` e `PyTorch Geometric`.
2.  **Simular um Conjunto de Dados de Grafo**: Criar um pequeno grafo sintético com nós (contas) e arestas (transações), onde alguns nós são marcados como fraudulentos.
3.  **Definir um Modelo GCN**: Implementar uma arquitetura GCN simples.
4.  **Treinar e Avaliar o Modelo**: Treinar o GCN para classificar nós como fraudulentos ou não fraudulentos.

**Observação**: Este é um exemplo simplificado com dados sintéticos. Em cenários reais, a construção do grafo, a engenharia de features e o tratamento de desbalanceamento de classes são muito mais complexos.

In [ ]:
# Instalação das bibliotecas necessárias
# PyTorch Geometric (PyG) é uma biblioteca para Deep Learning em Grafos
# Certifique-se de que o CUDA (GPU) esteja disponível se for usar para datasets maiores.

import torch

def check_torch_cuda():
    if torch.cuda.is_available():
        return torch.version.cuda
    else:
        return 'cpu'

cuda_version = check_torch_cuda()
print(f"PyTorch CUDA version: {cuda_version}")

# Adaptação da instalação para diferentes versões CUDA ou CPU
TORCH = ''
if cuda_version != 'cpu':
    TORCH = f"torch=={torch.__version__}+cu{cuda_version.replace('.', '')}"
else:
    TORCH = f"torch=={torch.__version__}"

pip_install_cmd = f"pip install {TORCH} -f https://download.pytorch.org/whl/torch_stable.html"
print(f"Executing: {pip_install_cmd}")
!{pip_install_cmd}

pip_install_pyg_cmd = f"pip install torch_geometric"
print(f"Executing: {pip_install_pyg_cmd}")
!{pip_install_pyg_cmd}

# Opcional: instala bibliotecas auxiliares, se necessário
pip_install_aux_cmd = f"pip install torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/{TORCH}.html"
print(f"Executing: {pip_install_aux_cmd}")
!{pip_install_aux_cmd}

print("Instalação concluída!")


PyTorch CUDA version: cpu
Executing: pip install torch==2.11.0+cpu -f https://download.pytorch.org/whl/torch_stable.html
Looking in links: https://download.pytorch.org/whl/torch_stable.html
Executing: pip install torch_geometric
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 66.7 MB/s eta 0:00:00
Executing: pip install torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch==2.11.0+cpu.html
Looking in links: https://data.pyg.org/whl/torch==2.11.0+cpu.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.0/108.0 kB 9.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.0/210.0 kB 20.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 5.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done


In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np

# 1. Simulação de um Conjunto de Dados de Grafo de Fraude
# Vamos criar um grafo simples onde alguns nós (contas) são fraudulentos.

num_nodes = 50 # Número total de contas (nós)
num_features = 5 # Número de características por conta (ex: volume médio de transações, idade da conta, etc.)
num_fraud_nodes = 5 # Número de contas fraudulentas

# Gerar características dos nós (features)
# Características para contas não fraudulentas (geralmente menores valores)
normal_features = torch.randn(num_nodes - num_fraud_nodes, num_features) * 0.5 + 1.0
# Características para contas fraudulentas (geralmente maiores valores ou diferentes padrões)
fraud_features = torch.randn(num_fraud_nodes, num_features) * 2.0 + 5.0

x = torch.cat((normal_features, fraud_features), dim=0).float()

# Criar rótulos (labels): 0 para não fraude, 1 para fraude
y = torch.zeros(num_nodes, dtype=torch.long)
y[-num_fraud_nodes:] = 1 # As últimas 'num_fraud_nodes' são marcadas como fraude

# Criar arestas (transações)
# Vamos criar um grafo aleatório, mas com maior conectividade entre nós do mesmo tipo
# e algumas conexões entre fraudulentos e não fraudulentos.

edge_index = []

# Conexões entre nós não fraudulentos
for i in range(num_nodes - num_fraud_nodes):
    for _ in range(np.random.randint(1, 4)): # 1 a 3 conexões aleatórias
        j = np.random.randint(num_nodes - num_fraud_nodes)
        if i != j:
            edge_index.append([i, j])

# Conexões entre nós fraudulentos
fraud_start_idx = num_nodes - num_fraud_nodes
for i in range(num_fraud_nodes):
    current_fraud_node = fraud_start_idx + i
    for _ in range(np.random.randint(1, 3)): # 1 a 2 conexões entre fraudes
        j = np.random.randint(num_fraud_nodes)
        other_fraud_node = fraud_start_idx + j
        if current_fraud_node != other_fraud_node:
            edge_index.append([current_fraud_node, other_fraud_node])

    # Algumas conexões de fraudes para não fraudes
    for _ in range(np.random.randint(1, 2)): # 1 conexão para não fraude
        j = np.random.randint(num_nodes - num_fraud_nodes)
        edge_index.append([current_fraud_node, j])
        # Adicionar também a transação inversa para grafo não direcionado
        edge_index.append([j, current_fraud_node])

edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()

# Remover arestas duplicadas e auto-loops
edge_index = torch_geometric.utils.coalesce(edge_index)

# Criar o objeto Data do PyTorch Geometric
data = Data(x=x, edge_index=edge_index, y=y)

print(f"Dados do Grafo:\n{data}")
print(f"Número de nós: {data.num_nodes}")
print(f"Número de arestas: {data.num_edges}")
print(f"Dimensão das features dos nós: {data.num_node_features}")
print(f"Nós fraudulentos (labels=1): {y.sum().item()}")
print(f"Nós não fraudulentos (labels=0): {(y == 0).sum().item()}")

# Visualização do Grafo (opcional, para grafos pequenos)
def visualize_graph(data):
    G = nx.Graph()
    G.add_nodes_from(range(data.num_nodes))
    edges = data.edge_index.t().tolist()
    G.add_edges_from(edges)

    plt.figure(figsize=(10, 8))
    node_colors = ['red' if label == 1 else 'blue' for label in data.y]
    nx.draw_networkx(G, with_labels=True, node_color=node_colors, node_size=300, font_size=8, alpha=0.8)
    plt.title('Grafo Sintético de Transações (Azul: Normal, Vermelho: Fraude)')
    plt.show()

# visualize_graph(data) # Descomente para visualizar, pode ser lento para muitos nós

# Divisão dos dados em treino e teste (máscaras)
# Para grafos, geralmente separamos nós para treinamento e teste
num_train = int(0.8 * num_nodes) # 80% para treino

train_mask = torch.zeros(num_nodes, dtype=torch.bool)
train_mask[:num_train] = True

test_mask = ~train_mask # O restante para teste

data.train_mask = train_mask
data.test_mask = test_mask

print(f"Nós para Treino: {data.train_mask.sum().item()}")
print(f"Nós para Teste: {data.test_mask.sum().item()}")


In [ ]:
# 2. Definição do Modelo GCN

class GCN(torch.nn.Module):
    def __init__(self, num_node_features, hidden_channels, num_classes):
        super().__init__()
        # Primeira camada GCN: mapeia as features de entrada para 'hidden_channels'
        self.conv1 = GCNConv(num_node_features, hidden_channels)
        # Segunda camada GCN: mapeia 'hidden_channels' para o número de classes de saída
        self.conv2 = GCNConv(hidden_channels, num_classes)

    def forward(self, x, edge_index):
        # x é a matriz de features dos nós (data.x)
        # edge_index é a matriz de conectividade (data.edge_index)

        # Aplica a primeira camada convolucional, seguida por ReLU e dropout
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)

        # Aplica a segunda camada convolucional
        x = self.conv2(x, edge_index)

        return x # Retorna os logits para cada classe

# Instancia o modelo
# num_node_features: dimensão das features de entrada (data.num_node_features)
# hidden_channels: número de neurônios na camada oculta
# num_classes: número de classes de saída (2 para fraude/não fraude)

model = GCN(num_node_features=data.num_node_features, hidden_channels=16, num_classes=2)
print(f"Modelo GCN:\n{model}")

# Mover o modelo para a GPU se disponível
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
data = data.to(device)


In [ ]:
# 3. Treinamento do Modelo GCN

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = torch.nn.CrossEntropyLoss() # Ideal para problemas de classificação multi-classe

def train():
    model.train() # Coloca o modelo em modo de treinamento
    optimizer.zero_grad() # Zera os gradientes

    # Executa o forward pass
    out = model(data.x, data.edge_index)
    # Calcula a perda apenas para os nós de treinamento
    loss = criterion(out[data.train_mask], data.y[data.train_mask])

    loss.backward() # Backpropagation: calcula os gradientes
    optimizer.step() # Otimização: atualiza os pesos do modelo
    return loss

def test():
    model.eval() # Coloca o modelo em modo de avaliação
    with torch.no_grad(): # Desativa o cálculo de gradientes
        out = model(data.x, data.edge_index)
        # Acha a classe prevista (índice com maior logit)
        pred = out.argmax(dim=1)
        # Calcula a acurácia para os nós de teste
        correct = (pred[data.test_mask] == data.y[data.test_mask]).sum()
        acc = int(correct) / int(data.test_mask.sum())
        return acc

num_epochs = 200
losses = []
accuracies = []

print("Iniciando treinamento...")
for epoch in range(1, num_epochs + 1):
    loss = train()
    acc = test()
    losses.append(loss.item())
    accuracies.append(acc)
    if epoch % 20 == 0:
        print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, Test Accuracy: {acc:.4f}')

print("Treinamento concluído!")

# Plotar a perda e a acurácia (opcional)
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(losses)
plt.title('Loss ao longo das Épocas')
plt.xlabel('Época')
plt.ylabel('Loss')

plt.subplot(1, 2, 2)
plt.plot(accuracies)
plt.title('Acurácia de Teste ao longo das Épocas')
plt.xlabel('Época')
plt.ylabel('Acurácia')
plt.show()


In [ ]:
# 4. Avaliação Final do Modelo

model.eval() # Coloca o modelo em modo de avaliação
with torch.no_grad():
    out = model(data.x, data.edge_index)
    pred = out.argmax(dim=1) # Previsões do modelo (0 ou 1)

# Calcular métricas para os dados de teste
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

y_true = data.y[data.test_mask].cpu().numpy()
y_pred = pred[data.test_mask].cpu().numpy()

print(f"\n--- Relatório de Avaliação no Conjunto de Teste ---")
print(f"Acurácia: {accuracy_score(y_true, y_pred):.4f}")
print(f"Precisão: {precision_score(y_true, y_pred):.4f}")
print(f"Recall (Sensibilidade): {recall_score(y_true, y_pred):.4f}")
print(f"F1-Score: {f1_score(y_true, y_pred):.4f}")

cm = confusion_matrix(y_true, y_pred)
print("\nMatriz de Confusão:")
print(cm)

print("\nInterpretação da Matriz de Confusão:")
print(f"  [[TN, FP],\n   [FN, TP]]")
print(f"  TN (Verdadeiros Negativos): {cm[0,0]} - Fraudes corretamente identificadas como não-fraude")
print(f"  FP (Falsos Positivos): {cm[0,1]} - Não-fraudes identificadas como fraude (alertas falsos)")
print(f"  FN (Falsos Negativos): {cm[1,0]} - Fraudes não detectadas (o mais crítico!)")
print(f"  TP (Verdadeiros Positivos): {cm[1,1]} - Fraudes corretamente detectadas")

# Exemplo de predições para alguns nós (incluindo fraudulentos)
print("\nPredições para alguns nós de teste (Índice | Real | Predito):")
for i, (true_label, predicted_label) in enumerate(zip(y_true, y_pred)):
    if i < 10: # Mostrar os primeiros 10 do conjunto de teste
        print(f"  Nó {data.test_mask.nonzero(as_tuple=True)[0][i].item():<3} | {true_label} | {predicted_label}")

print("\nEste exemplo demonstra como uma GCN pode ser estruturada para a detecção de fraude em dados de grafos. Em um cenário real, você precisaria de um dataset de grafo de transações muito maior e mais complexo, além de tunar o modelo e explorar features mais ricas.")
